# 🌍 AirSentinel Cameroun
## Notebook 05 — IRS par ACP + Épisodes climatiques
**Responsable : BODEHOU Sabine — ISSEA**

### Objectif
1. Calculer l'IRS par ACP sur les 36 moyennes de villes
2. Définir les seuils par percentiles 50/75/90
3. Valider avec les seuils OMS 2021

### Références
- WHO AQG 2021 — NCBI NBK574591
- Ceccherini et al. 2017 — percentile 90

In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import joblib, warnings
warnings.filterwarnings('ignore')

df = pd.read_excel('../data/processed/dataset_enrichi.xlsx')
df['date'] = pd.to_datetime(df['date'])
print(f'✅ Dataset chargé : {df.shape[0]:,} lignes')

✅ Dataset chargé : 44,415 lignes


## Étape 1 — ACP sur les 36 moyennes de villes
**Pourquoi les moyennes ?**
Les 36 villes ont des profils climatiques indépendants.
Les données journalières se ressemblent d'un jour à l'autre (autocorrélation).
→ On prend 1 moyenne par ville = 36 lignes indépendantes ✅

In [2]:
cols_irs = ['pm2_5_moyen', 'dust_moyen', 'co_moyen', 'uv_moyen', 'ozone_moyen']
cols_irs = [c for c in cols_irs if c in df.columns]

# Moyennes par ville (36 lignes indépendantes)
df_moyennes = df.groupby('ville')[cols_irs].mean().dropna()
print(f'ACP sur {len(df_moyennes)} profils de villes indépendants')

# Normalisation
scaler_acp = StandardScaler()
X_moy      = scaler_acp.fit_transform(df_moyennes)

# ACP
pca_test = PCA()
pca_test.fit(X_moy)

print('\nVariance expliquée par composante :')
cumul = 0
for i, var in enumerate(pca_test.explained_variance_ratio_):
    cumul += var
    print(f'  Axe {i+1} : {var*100:.1f}%  → cumulé : {cumul*100:.1f}%')

ACP sur 35 profils de villes indépendants

Variance expliquée par composante :
  Axe 1 : 55.2%  → cumulé : 55.2%
  Axe 2 : 27.9%  → cumulé : 83.1%
  Axe 3 : 9.1%  → cumulé : 92.2%
  Axe 4 : 7.4%  → cumulé : 99.6%
  Axe 5 : 0.4%  → cumulé : 100.0%


In [3]:
# Choisir le nombre d'axes selon variance expliquée
variance_cumulee = np.cumsum(pca_test.explained_variance_ratio_)
n_axes = np.argmax(variance_cumulee >= 0.80) + 1
print(f'Nombre d axes retenus : {n_axes} (expliquent {variance_cumulee[n_axes-1]*100:.1f}% de la variance)')

# ACP finale
pca = PCA(n_components=n_axes)
pca.fit(X_moy)

print('\nPoids IRS calculés par ACP :')
if n_axes == 1:
    poids = abs(pca.components_[0])
    poids = poids / poids.sum()
    for col, p in zip(cols_irs, poids):
        print(f'  {col:20s} : {p:.4f} ({p*100:.1f}%)')
else:
    v1 = pca.explained_variance_ratio_[0]
    v2 = pca.explained_variance_ratio_[1]
    total = v1 + v2
    print(f'  Axe 1 ({v1*100:.1f}%) : ', dict(zip(cols_irs, abs(pca.components_[0]).round(3))))
    print(f'  Axe 2 ({v2*100:.1f}%) : ', dict(zip(cols_irs, abs(pca.components_[1]).round(3))))

Nombre d axes retenus : 2 (expliquent 83.1% de la variance)

Poids IRS calculés par ACP :
  Axe 1 (55.2%) :  {'pm2_5_moyen': np.float64(0.528), 'dust_moyen': np.float64(0.472), 'co_moyen': np.float64(0.225), 'uv_moyen': np.float64(0.438), 'ozone_moyen': np.float64(0.506)}
  Axe 2 (27.9%) :  {'pm2_5_moyen': np.float64(0.344), 'dust_moyen': np.float64(0.387), 'co_moyen': np.float64(0.78), 'uv_moyen': np.float64(0.35), 'ozone_moyen': np.float64(0.041)}


## Étape 2 — Calculer l'IRS

In [4]:
X_full = scaler_acp.transform(df[cols_irs].fillna(df[cols_irs].median()))
scores = pca.transform(X_full)

if n_axes == 1:
    df['IRS_brut'] = scores[:, 0]
else:
    v1 = pca.explained_variance_ratio_[0]
    v2 = pca.explained_variance_ratio_[1]
    total = v1 + v2
    df['IRS_brut'] = (v1/total) * scores[:, 0] + (v2/total) * scores[:, 1]

# Normaliser entre 0 et 1
irs_min = df['IRS_brut'].min()
irs_max = df['IRS_brut'].max()
df['IRS'] = (df['IRS_brut'] - irs_min) / (irs_max - irs_min)
print(f'IRS calculé — Moyenne : {df["IRS"].mean():.3f}')

IRS calculé — Moyenne : 0.090


## Étape 3 — Seuils par percentiles
**Pourquoi les percentiles ?** Défendables devant les juges.
CRITIQUE = pire que 90% des jours observés au Cameroun.

In [5]:
p50 = df['IRS'].quantile(0.50)
p75 = df['IRS'].quantile(0.75)
p90 = df['IRS'].quantile(0.90)

print('Seuils IRS calculés depuis vos données :')
print(f'  🟢 FAIBLE   : IRS < {p50:.3f}  (journée normale)')
print(f'  🟡 MODÉRÉ   : {p50:.3f} ≤ IRS < {p75:.3f}')
print(f'  🟠 ÉLEVÉ    : {p75:.3f} ≤ IRS < {p90:.3f}')
print(f'  🔴 CRITIQUE : IRS ≥ {p90:.3f}  (pire 10%)')

# Validation OMS — WHO AQG 2021, NCBI NBK574591, Table 3.6
SEUIL_OMS = 15
pm25_crit = df.loc[df['IRS'] >= p90, 'pm2_5_moyen'].mean()
print(f'\nValidation OMS : PM2.5 quand CRITIQUE = {pm25_crit:.1f} µg/m³')
print(f'Seuil OMS = {SEUIL_OMS} µg/m³')
print('✅ Cohérent' if pm25_crit > SEUIL_OMS else '⚠️ Vérifier')

df['niveau_sanitaire'] = df['IRS'].apply(
    lambda x: '🟢 FAIBLE' if x < p50
         else '🟡 MODÉRÉ' if x < p75
         else '🟠 ÉLEVÉ'  if x < p90
         else '🔴 CRITIQUE' if pd.notna(x) else 'N/A'
)

Seuils IRS calculés depuis vos données :
  🟢 FAIBLE   : IRS < 0.075  (journée normale)
  🟡 MODÉRÉ   : 0.075 ≤ IRS < 0.116
  🟠 ÉLEVÉ    : 0.116 ≤ IRS < 0.162
  🔴 CRITIQUE : IRS ≥ 0.162  (pire 10%)

Validation OMS : PM2.5 quand CRITIQUE = 53.6 µg/m³
Seuil OMS = 15 µg/m³
✅ Cohérent


## Étape 4 — Sauvegarder

In [6]:
joblib.dump(scaler_acp,      '../models/scaler_acp_irs.pkl')
joblib.dump(pca,             '../models/pca_irs.pkl')
joblib.dump(cols_irs,        '../models/cols_irs.pkl')
joblib.dump({'p50':p50,'p75':p75,'p90':p90,'irs_min':irs_min,'irs_max':irs_max},
            '../models/seuils_irs.pkl')

df.to_excel('../data/processed/dataset_final.xlsx', index=False)
print('✅ IRS calculé et sauvegardé')
print('✅ Dataset final sauvegardé : data/processed/dataset_final.xlsx')

✅ IRS calculé et sauvegardé
✅ Dataset final sauvegardé : data/processed/dataset_final.xlsx
